In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import gc
import json
import sys
import zipfile
from collections import Counter
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
from numpy.random import SeedSequence
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_auc_score,
)
from tqdm import tqdm

from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
experiment_name = "lsa_convtok"

drive_project_path   = "/content/drive/MyDrive/colab/lsa_convtok_vit"
drive_scripts_path   = os.path.join(drive_project_path, "scripts")
drive_preprocessed_zip = os.path.join(
    drive_project_path, "preprocessed", "preprocessed_384.zip"
)
drive_split_indices  = os.path.join(
    drive_project_path, "preprocessed", "split_indices.json"
)

exp_suffix           = experiment_name
drive_output_path    = os.path.join(drive_project_path, "output",      exp_suffix)
drive_checkpoint_path = os.path.join(drive_project_path, "checkpoints", exp_suffix)

local_preprocessed_path = "/content/data"
local_scripts_path       = "/content/scripts"
local_checkpoint_path    = os.path.join("/content/checkpoints", exp_suffix)

for p in [
    local_preprocessed_path,
    local_scripts_path,
    local_checkpoint_path,
    drive_output_path,
    drive_checkpoint_path,
]:
    os.makedirs(p, exist_ok=True)

get_ipython().system("pip install einops --quiet")
get_ipython().system(f"cp -r {drive_scripts_path}/* {local_scripts_path}/")
sys.path.insert(0, local_scripts_path)

In [3]:
from checkpoints import CheckpointManager, initialize_master_state
from config import get_config, save_config
from plotting_utils import TrainingPlotter
from training_logic import ModelTrainer, determine_optimal_epochs
from training_utils import (
    NPYImageFolder,
    aggregate_predictions_by_group,
    calculate_class_weights,
    calculate_dataset_stats,
    create_data_loaders,
    create_slide_level_folds,
    create_transforms,
    get_device_info,
    grouped_bootstrap_ci,
    print_data_distribution,
    verify_data_splits,
)

In [4]:
config = get_config(experiment_name)
save_config(config, drive_output_path)

n_iterations = int(config["n_iterations"])
n_folds      = int(config["n_folds"])
base_seed    = int(config["random_seed"])

checkpoint_manager = CheckpointManager(
    local_checkpoint_dir=local_checkpoint_path,
    drive_checkpoint_dir=drive_checkpoint_path,
    n_folds=n_folds,
)
plotter = TrainingPlotter(drive_output_path, dpi=300)

checkpoint_manager.cleanup_temp_files(verbose=True)
checkpoint_manager.sync_from_drive(verbose=True)

device = get_device_info()

# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------
preprocessed_dir = os.path.join(local_preprocessed_path, "preprocessed")

if not os.path.exists(preprocessed_dir):
    if not os.path.exists(drive_preprocessed_zip):
        raise FileNotFoundError(
            f"Preprocessed data not found: {drive_preprocessed_zip}"
        )
    with zipfile.ZipFile(drive_preprocessed_zip, "r") as zf:
        zf.extractall(local_preprocessed_path)
else:
    print(f"Using existing preprocessed data: {preprocessed_dir}")

metadata: Dict = {}
metadata_path  = os.path.join(preprocessed_dir, "metadata.json")
if os.path.exists(metadata_path):
    with open(metadata_path, "r") as f:
        metadata = json.load(f)
    print(f"Loaded metadata from: {os.path.basename(metadata_path)}")
else:
    print("WARNING: metadata.json not found; using minimal metadata.")
    metadata = {
        "method": "unknown",
        "bit_depth": None,
        "image_size": config.get("image_size"),
        "n_channels": config.get("channels", 1),
    }

# Consistency checks
for field, cfg_key, meta_key in [
    ("bit_depth",  "bit_depth",  "bit_depth"),
    ("image_size", "image_size", "image_size"),
]:
    cfg_val  = config.get(cfg_key)
    meta_val = metadata.get(meta_key)
    if cfg_val is not None and meta_val is not None and cfg_val != meta_val:
        print(
            f"WARNING: config {field}={cfg_val} but "
            f"preprocessing {field}={meta_val}."
        )

  Config saved to: /content/drive/MyDrive/colab/lsa_convtok_vit/output/lsa_convtok/config_lsa_convtok.json
  Synced: cv_iter_29_fold_2_checkpoint.pth
  Synced: cv_iter_29_fold_3_checkpoint.pth
  Synced: master_training_state.json
Sync complete: 3 file(s) updated

Device Information:
  Device: cuda
  GPU: Tesla T4
  Memory: 15.64 GB
  CUDA Version: 12.8
Loaded metadata from: metadata.json


In [5]:
# ---------------------------------------------------------------------------
# Split indices
# ---------------------------------------------------------------------------
if not os.path.exists(drive_split_indices):
    raise FileNotFoundError(
        f"Split indices not found: {drive_split_indices}"
    )
with open(drive_split_indices, "r") as f:
    split_data = json.load(f)

train_base_paths = [Path(p) for p in split_data["train_base_paths"]]
test_base_paths  = [Path(p) for p in split_data["test_base_paths"]]
class_names      = split_data["classes"]

print(f"Classes      : {class_names}")
print(f"Train images : {len(train_base_paths)}")
print(f"Test images  : {len(test_base_paths)}")

dataset     = NPYImageFolder(preprocessed_dir)
num_classes = len(dataset.classes)
labels      = np.array(dataset.targets)

print(f"Total images : {len(dataset)}")
print(f"Num classes  : {num_classes}")
print(f"Dataset classes: {dataset.classes}")

# Build index maps
npy_base_to_idx: Dict[str, int] = {}
idx_to_base: Dict[int, Path]    = {}

for idx, (sample_path, _) in enumerate(dataset.samples):
    rel_npy  = Path(sample_path).relative_to(preprocessed_dir)
    base_npy = rel_npy.with_suffix("")
    npy_base_to_idx[base_npy.as_posix()] = idx
    idx_to_base[idx] = Path(base_npy)

train_val_idx: List[int] = []
test_idx:      List[int] = []

for base in train_base_paths:
    key = base.as_posix()
    if key not in npy_base_to_idx:
        raise RuntimeError(f"Train path not in dataset: {key}")
    train_val_idx.append(npy_base_to_idx[key])

for base in test_base_paths:
    key = base.as_posix()
    if key not in npy_base_to_idx:
        raise RuntimeError(f"Test path not in dataset: {key}")
    test_idx.append(npy_base_to_idx[key])

print(f"TrainVal: {len(train_val_idx)} | Test: {len(test_idx)}")

verify_data_splits(train_val_idx, [], test_idx)
print_data_distribution(labels[train_val_idx].tolist(), dataset.classes, "TrainVal")
print_data_distribution(labels[test_idx].tolist(),      dataset.classes, "Test")

Classes      : ['Candida albicans', 'Candida glabrata']
Train images : 2910
Test images  : 821
NPYImageFolder initialized: 3763 samples (grayscale)
Total images : 3763
Num classes  : 2
Dataset classes: ['Candida albicans', 'Candida glabrata']
TrainVal: 2910 | Test: 821
✓ Data splits verified: No overlap detected

TrainVal Distribution:
  Total samples: 2910
  Candida albicans: 1616 (55.5%)
  Candida glabrata: 1294 (44.5%)

Test Distribution:
  Total samples: 821
  Candida albicans: 440 (53.6%)
  Candida glabrata: 381 (46.4%)


In [6]:
# ---------------------------------------------------------------------------
# Slide ID extraction
# ---------------------------------------------------------------------------

def extract_slide_id_from_base(p: Path) -> str:
    stem  = p.name
    parts = stem.split("_")
    if len(parts) < 3:
        return stem
    return f"{parts[0]}_{parts[1]}_{parts[2]}"


all_groups = np.empty(len(dataset), dtype=object)
for idx in range(len(dataset)):
    all_groups[idx] = extract_slide_id_from_base(idx_to_base[idx])

test_groups = all_groups[test_idx]


# ---------------------------------------------------------------------------
# Phase: Cross-validation
# ---------------------------------------------------------------------------

def run_cross_validation(master_state: Dict) -> None:
    """30-iteration × 5-fold slide-stratified cross-validation."""
    print("-" * 70)
    print(f"PHASE: CROSS-VALIDATION [{experiment_name}] (Slide-Level Stratification)")
    print("-" * 70)
    print(f"Base seed: {base_seed}")

    start_iteration = int(master_state.get("current_iteration", 0))
    start_fold      = int(master_state.get("current_fold", 0))
    cv_fold_details = list(master_state.get("cv_fold_details", []))

    trainer = ModelTrainer(config, device, checkpoint_manager, num_classes)

    ss                  = SeedSequence(base_seed)
    iteration_seed_gens = ss.spawn(n_iterations)

    for iter_idx in range(start_iteration, n_iterations):
        iteration_seed = int(
            iteration_seed_gens[iter_idx].generate_state(1)[0]
        )
        print(f"\n{'='*70}")
        print(f"ITERATION {iter_idx+1}/{n_iterations} | seed={iteration_seed}")
        print(f"{'='*70}")

        # Slide-level stratified folds
        train_val_groups = all_groups[train_val_idx]
        fold_indices, _ = create_slide_level_folds(
            image_indices=np.array(train_val_idx),
            image_labels=labels,
            slide_groups=all_groups,
            n_splits=n_folds,
            random_state=iteration_seed,
            class_names=class_names,
        )

        for fold_idx, (fold_train_idx, fold_val_idx) in enumerate(fold_indices):
            if iter_idx == start_iteration and fold_idx < start_fold:
                continue

            print(f"\n{'-'*70}")
            print(f"FOLD {fold_idx+1}/{n_folds}")

            val_groups_fold = all_groups[fold_val_idx]

            # Per-fold dataset statistics (train set only)
            fold_mean, fold_std = calculate_dataset_stats(
                dataset, list(fold_train_idx)
            )

            train_transform = create_transforms(
                fold_mean, fold_std, config, train=True
            )
            val_transform   = create_transforms(
                fold_mean, fold_std, config, train=False
            )

            train_loader, val_loader = create_data_loaders(
                dataset,
                fold_train_idx,
                fold_val_idx,
                train_transform,
                val_transform,
                batch_size=config['batch_size'],
                num_workers=config['num_workers'],
            )

            class_weights = calculate_class_weights(
                labels[fold_train_idx].tolist(),
                num_classes,
                device,
                gamma=config.get('class_weight_gamma', 0.0),
            )

            fold_summary = trainer.train_cv_fold(
                iter_idx=iter_idx,
                fold_idx=fold_idx,
                iteration_seed=iteration_seed,
                train_loader=train_loader,
                val_loader=val_loader,
                fold_mean=fold_mean,
                fold_std=fold_std,
                class_weights=class_weights,
                val_groups_fold=val_groups_fold,
            )

            cv_fold_details.append(fold_summary)

            master_state['current_iteration']  = iter_idx
            master_state['current_fold']        = fold_idx + 1
            master_state['cv_fold_details']     = cv_fold_details
            checkpoint_manager.save_master_state(
                master_state, backup_to_drive=True
            )

            gc.collect()

        master_state['current_fold'] = 0

    master_state['phase']                    = 'analysis'
    master_state['all_iterations_completed'] = True
    checkpoint_manager.save_master_state(master_state, backup_to_drive=True)
    print("\nCross-validation complete.")

In [7]:
# ---------------------------------------------------------------------------
# Phase: Analysis
# ---------------------------------------------------------------------------

def run_analysis(master_state: Dict) -> None:
    """Compute CV statistics and determine optimal training epochs."""
    print("-" * 70)
    print(f"PHASE: ANALYSIS [{experiment_name}]")
    print("-" * 70)

    cv_fold_details = master_state.get('cv_fold_details', [])
    if not cv_fold_details:
        raise RuntimeError("No CV fold details found in master state.")

    cv_results_df = pd.DataFrame(cv_fold_details)
    results_path  = checkpoint_manager.get_results_csv_path(experiment_name)
    cv_results_df.to_csv(results_path, index=False)
    print(f"CV results saved: {results_path}")

    plotter.plot_iteration_results(
        cv_results_df,
        experiment_name=experiment_name,
        save_name=f"cv_iteration_results_{experiment_name}.png",
        show=False,
    )

    optimal_epochs = determine_optimal_epochs(
        cv_fold_details=cv_fold_details,
        method="percentile_75",
        config=config,
        verbose=True,
    )

    master_state['optimal_epochs'] = optimal_epochs
    master_state['phase']           = 'final_training'
    checkpoint_manager.save_master_state(master_state, backup_to_drive=True)

In [8]:
# ---------------------------------------------------------------------------
# Phase: Final training
# ---------------------------------------------------------------------------

def run_final_training(master_state: Dict) -> None:
    """Train final model on all training data for optimal_epochs."""
    print("-" * 70)
    print(f"PHASE: FINAL TRAINING [{experiment_name}]")
    print("-" * 70)

    optimal_epochs = master_state.get('optimal_epochs', config['num_epochs'])
    print(f"Optimal epochs: {optimal_epochs}")

    final_mean, final_std = calculate_dataset_stats(
        dataset, list(train_val_idx)
    )

    train_transform = create_transforms(final_mean, final_std, config, train=True)
    train_loader, _ = create_data_loaders(
        dataset,
        train_val_idx,
        [],
        train_transform,
        None,
        batch_size=config['batch_size'],
        num_workers=config['num_workers'],
    )

    class_weights = calculate_class_weights(
        labels[train_val_idx].tolist(),
        num_classes,
        device,
        gamma=config.get('class_weight_gamma', 0.0),
    )

    trainer = ModelTrainer(config, device, checkpoint_manager, num_classes)
    final_model, final_history = trainer.train_final_model(
        train_loader=train_loader,
        optimal_epochs=optimal_epochs,
        final_mean=final_mean,
        final_std=final_std,
        class_weights=class_weights,
    )

    plotter.plot_training_curves(
        final_history,
        title=f"Final Model Training — {experiment_name}",
        save_name=f"training_curves_{experiment_name}.png",
        show=False,
    )

    master_state['final_mean'] = final_mean
    master_state['final_std']  = final_std
    master_state['phase']       = 'evaluation'
    checkpoint_manager.save_master_state(master_state, backup_to_drive=True)

In [9]:
# ---------------------------------------------------------------------------
# Phase: Evaluation
# ---------------------------------------------------------------------------

def run_evaluation(master_state: Dict) -> None:
    """Evaluate the final model on the held-out test set."""
    print("-" * 70)
    print(f"PHASE: EVALUATION [{experiment_name}]")
    print("-" * 70)

    final_mean = master_state.get('final_mean', [0.0])
    final_std  = master_state.get('final_std',  [1.0])

    # Load final checkpoint
    checkpoint = checkpoint_manager.load_checkpoint(
        checkpoint_manager.get_final_checkpoint_path(use_local=False),
        verbose=True,
    )
    if checkpoint is None:
        checkpoint = checkpoint_manager.load_checkpoint(
            checkpoint_manager.get_final_checkpoint_path(use_local=True),
            verbose=True,
        )
    if checkpoint is None:
        raise RuntimeError("Final model checkpoint not found.")

    trainer = ModelTrainer(config, device, checkpoint_manager, num_classes)
    model   = trainer.create_model(model_seed=config['random_seed'])
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    test_transform = create_transforms(final_mean, final_std, config, train=False)
    _, test_loader = create_data_loaders(
        dataset,
        [],
        test_idx,
        None,
        test_transform,
        batch_size=config['batch_size'],
        num_workers=config['num_workers'],
    )

    all_labels  = []
    all_preds   = []
    all_probs   = []

    with torch.inference_mode():
        for X, y in tqdm(test_loader, desc="Evaluating"):
            X = X.to(device)
            logits = model(X)
            probs  = logits.softmax(dim=1)
            preds  = logits.argmax(dim=1)
            all_labels.extend(y.numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)

    # Image-level metrics
    test_bal_acc = balanced_accuracy_score(all_labels, all_preds)
    test_mcc     = matthews_corrcoef(all_labels, all_preds)
    test_f1      = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    test_auc     = roc_auc_score(all_labels, all_probs[:, 1])

    # Bootstrap CIs (grouped by slide for test-set validity)
    bal_acc_ci = grouped_bootstrap_ci(balanced_accuracy_score, [all_labels, all_preds], test_groups)
    mcc_ci     = grouped_bootstrap_ci(matthews_corrcoef,       [all_labels, all_preds], test_groups)
    f1_ci      = grouped_bootstrap_ci(
        lambda yt, yp: f1_score(yt, yp, average='weighted', zero_division=0),
        [all_labels, all_preds], test_groups
    )
    auc_ci     = grouped_bootstrap_ci(
        lambda yt, yp: roc_auc_score(yt, yp[:, 1]),
        [all_labels, all_probs], test_groups
    )

    # Slide-level aggregation
    slide_true, slide_pred, slide_proba, _ = aggregate_predictions_by_group(
        all_labels, all_preds, all_probs, test_groups, num_classes
    )
    slide_bal_acc = balanced_accuracy_score(slide_true, slide_pred)
    slide_mcc     = matthews_corrcoef(slide_true, slide_pred)
    slide_f1      = f1_score(slide_true, slide_pred, average='weighted', zero_division=0)
    slide_auc     = roc_auc_score(slide_true, slide_proba[:, 1])

    # For slide-level CIs each slide is its own observation — use integer indices as groups
    slide_groups_idx = np.arange(len(slide_true))
    slide_bal_acc_ci = grouped_bootstrap_ci(balanced_accuracy_score, [slide_true, slide_pred], slide_groups_idx)
    slide_mcc_ci     = grouped_bootstrap_ci(matthews_corrcoef,       [slide_true, slide_pred], slide_groups_idx)
    slide_f1_ci      = grouped_bootstrap_ci(
        lambda yt, yp: f1_score(yt, yp, average='weighted', zero_division=0),
        [slide_true, slide_pred], slide_groups_idx
    )
    slide_auc_ci     = grouped_bootstrap_ci(
        lambda yt, yp: roc_auc_score(yt, yp[:, 1]),
        [slide_true, slide_proba], slide_groups_idx
    )

    # Per-class metrics
    p_img, r_img, f1_img, s_img = precision_recall_fscore_support(
        all_labels, all_preds, labels=list(range(num_classes)),
        average=None, zero_division=0,
    )
    p_slide, r_slide, f1_slide, s_slide = precision_recall_fscore_support(
        slide_true, slide_pred, labels=list(range(num_classes)),
        average=None, zero_division=0,
    )

    per_class_image = {}
    per_class_slide = {}
    for i, cn in enumerate(dataset.classes):
        per_class_image[cn] = {
            "precision": float(p_img[i]), "recall": float(r_img[i]),
            "f1_score": float(f1_img[i]), "support": int(s_img[i]),
        }
        per_class_slide[cn] = {
            "precision": float(p_slide[i]), "recall": float(r_slide[i]),
            "f1_score": float(f1_slide[i]), "support": int(s_slide[i]),
        }

    # Print summary
    print(f"\n{'='*70}")
    print(f"TEST SET RESULTS — {experiment_name.upper()}")
    print(f"{'='*70}")
    print("\nIMAGE-LEVEL:")
    print(f"  Balanced Acc : {test_bal_acc:.4f}  CI=[{bal_acc_ci[0]:.4f}, {bal_acc_ci[1]:.4f}]")
    print(f"  MCC          : {test_mcc:.4f}      CI=[{mcc_ci[0]:.4f}, {mcc_ci[1]:.4f}]")
    print(f"  F1 (weighted): {test_f1:.4f}        CI=[{f1_ci[0]:.4f}, {f1_ci[1]:.4f}]")
    print(f"  AUC          : {test_auc:.4f}       CI=[{auc_ci[0]:.4f}, {auc_ci[1]:.4f}]")
    print("\nSLIDE-LEVEL:")
    print(f"  Balanced Acc : {slide_bal_acc:.4f}  CI=[{slide_bal_acc_ci[0]:.4f}, {slide_bal_acc_ci[1]:.4f}]")
    print(f"  MCC          : {slide_mcc:.4f}      CI=[{slide_mcc_ci[0]:.4f}, {slide_mcc_ci[1]:.4f}]")
    print(f"  F1 (weighted): {slide_f1:.4f}        CI=[{slide_f1_ci[0]:.4f}, {slide_f1_ci[1]:.4f}]")
    print(f"  AUC          : {slide_auc:.4f}       CI=[{slide_auc_ci[0]:.4f}, {slide_auc_ci[1]:.4f}]")
    print(f"\n{'='*70}")

    # Visualisations
    plotter.plot_confusion_matrix(
        all_labels, all_preds, dataset.classes,
        title=f"Confusion Matrix (Image) — {experiment_name}",
        save_name=f"confusion_matrix_image_{experiment_name}.png",
        normalize=True, show=False,
    )
    plotter.plot_confusion_matrix(
        slide_true, slide_pred, dataset.classes,
        title=f"Confusion Matrix (Slide) — {experiment_name}",
        save_name=f"confusion_matrix_slide_{experiment_name}.png",
        normalize=True, show=False,
    )
    plotter.plot_roc_auc_curves(
        all_labels, all_probs,
        title=f"ROC Curve (Image) — {experiment_name}",
        save_name=f"roc_auc_image_{experiment_name}.png", show=False,
    )
    plotter.plot_roc_auc_curves(
        slide_true, slide_proba,
        title=f"ROC Curve (Slide) — {experiment_name}",
        save_name=f"roc_auc_slide_{experiment_name}.png", show=False,
    )

    # Save JSON results
    results = {
        "experiment_name": experiment_name,
        "metadata":        metadata,
        "overall_metrics_image_level": {
            "balanced_accuracy": float(test_bal_acc),
            "mcc":               float(test_mcc),
            "f1_score_weighted": float(test_f1),
            "auc_binary":        float(test_auc),
        },
        "confidence_intervals_95_image_level": {
            "balanced_accuracy": (float(bal_acc_ci[0]), float(bal_acc_ci[1])),
            "mcc":               (float(mcc_ci[0]),     float(mcc_ci[1])),
            "f1_score_weighted": (float(f1_ci[0]),      float(f1_ci[1])),
            "auc_binary":        (float(auc_ci[0]),     float(auc_ci[1])),
        },
        "overall_metrics_slide_level": {
            "balanced_accuracy": float(slide_bal_acc),
            "mcc":               float(slide_mcc),
            "f1_score_weighted": float(slide_f1),
            "auc_binary":        float(slide_auc),
        },
        "confidence_intervals_95_slide_level": {
            "balanced_accuracy": (float(slide_bal_acc_ci[0]), float(slide_bal_acc_ci[1])),
            "mcc":               (float(slide_mcc_ci[0]),     float(slide_mcc_ci[1])),
            "f1_score_weighted": (float(slide_f1_ci[0]),      float(slide_f1_ci[1])),
            "auc_binary":        (float(slide_auc_ci[0]),     float(slide_auc_ci[1])),
        },
        "per_class_metrics_image_level": per_class_image,
        "per_class_metrics_slide_level": per_class_slide,
        "test_set_size_images":          len(test_idx),
        "test_set_size_slides":          int(len(np.unique(test_groups))),
        "unique_test_slides":            sorted([str(s) for s in np.unique(test_groups)]),
    }

    results_path = os.path.join(
        drive_output_path, f"test_results_{experiment_name}.json"
    )
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=4)
    print(f"\nResults saved: {os.path.basename(results_path)}")

    checkpoint_manager.cleanup_fold_histories(verbose=True)
    master_state['phase'] = 'completed'
    checkpoint_manager.save_master_state(master_state, backup_to_drive=True)
    print(f"\n{'='*70}\nEVALUATION COMPLETE\n{'='*70}\n")

In [10]:
# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main() -> None:
    master_state = checkpoint_manager.load_master_state(prefer_drive=True)
    if master_state is None:
        print("No previous state found. Initialising...")
        master_state = initialize_master_state()
        checkpoint_manager.save_master_state(master_state, backup_to_drive=True)
    else:
        print(f"Resuming | phase={master_state.get('phase', 'unknown')}")

    try:
        phase = master_state.get('phase', 'cross_validation')

        if phase == 'cross_validation':
            run_cross_validation(master_state)
            phase = 'analysis'

        if phase == 'analysis':
            run_analysis(master_state)
            phase = 'final_training'

        if phase == 'final_training':
            run_final_training(master_state)
            phase = 'evaluation'

        if phase == 'evaluation':
            run_evaluation(master_state)
            phase = 'completed'

        if phase == 'completed':
            print("-" * 70)
            print(f"PIPELINE COMPLETED — {experiment_name}")
            print(f"Results: {drive_output_path}")
            print("-" * 70)

    except KeyboardInterrupt:
        print("\nInterrupted by user. State saved — re-run to resume.")
    except Exception as e:
        import traceback
        print(f"\nERROR: {e}")
        traceback.print_exc()
        print(f"Phase: {master_state.get('phase', 'unknown')}")
        print("Fix the error and re-run to resume from this phase.")


if __name__ == "__main__":
    main()

Loaded master state from: master_training_state.json
Resuming | phase=cross_validation
----------------------------------------------------------------------
PHASE: CROSS-VALIDATION [lsa_convtok] (Slide-Level Stratification)
----------------------------------------------------------------------
Base seed: 42
MixUp enabled: alpha=0.2, prob=0.5

ITERATION 30/30 | seed=911875363

✓ Valid splits found on first attempt

----------------------------------------------------------------------
FOLD 4/5
Using uniform class weights (gamma=0, recommended with MixUp)

[LSA_CONVTOK] ITERATION 30/30 | FOLD 4/5
Iteration seed: 911875363
Validation: ~9 slides, 500 patches
Resuming from epoch 24 | best epoch: 5
Model init seed: 259627004
  Model config:
    experiment       : lsa_convtok
    init_policy      : default
    dim_head         : 64
    pe_temperature   : 10000
    convtok_hidden_ch: 64
    arch_hash        : 95ace50634b621d7
[LSA-ConvTok] Analytical output grid: (24, 24) (N=576) for 384×384 

/content/scripts/plotting_utils.py:112: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="Metric", y="Mean Score", data=melted, palette="viridis", ax=ax)


CV iteration distribution plot saved to: /content/drive/MyDrive/colab/lsa_convtok_vit/output/lsa_convtok/cv_iteration_results_lsa_convtok.png

OPTIMAL EPOCHS CALCULATION
Input: 150 valid folds from 30 iterations | method: percentile_75
  Raw value : 43.0 → bounded: 43 epochs
  Range     : [5, 200]

  Train final model for 43 epochs

----------------------------------------------------------------------
PHASE: FINAL TRAINING [lsa_convtok]
----------------------------------------------------------------------
Optimal epochs: 43
Using uniform class weights (gamma=0, recommended with MixUp)
MixUp enabled: alpha=0.2, prob=0.5

[LSA_CONVTOK] TRAINING FINAL MODEL FOR 43 EPOCHS
Starting final training from scratch
  Model config:
    experiment       : lsa_convtok
    init_policy      : default
    dim_head         : 64
    pe_temperature   : 10000
    convtok_hidden_ch: 64
    arch_hash        : 95ace50634b621d7
[LSA-ConvTok] Analytical output grid: (24, 24) (N=576) for 384×384 input
[LSA-Con

Evaluating:   0%|          | 0/13 [00:00<?, ?it/s]

[LSA-ConvTok] Runtime token grid (forward #1): H'=24, W'=24, N=576
[LSA-ConvTok] Matched-N assertion passed: (24, 24) == (24, 24), N=576


Evaluating: 100%|██████████| 13/13 [00:06<00:00,  1.91it/s]



TEST SET RESULTS — LSA_CONVTOK

IMAGE-LEVEL:
  Balanced Acc : 0.9576  CI=[0.8521, 0.9981]
  MCC          : 0.9129      CI=[0.6822, 0.9957]
  F1 (weighted): 0.9550        CI=[0.8471, 0.9978]
  AUC          : 0.9832       CI=[0.9189, 1.0000]

SLIDE-LEVEL:
  Balanced Acc : 0.9286  CI=[0.7778, 1.0000]
  MCC          : 0.8452      CI=[0.5222, 1.0000]
  F1 (weighted): 0.9172        CI=[0.7404, 1.0000]
  AUC          : 1.0000       CI=[1.0000, 1.0000]

Confusion matrix saved to: /content/drive/MyDrive/colab/lsa_convtok_vit/output/lsa_convtok/confusion_matrix_image_lsa_convtok.png
Confusion matrix saved to: /content/drive/MyDrive/colab/lsa_convtok_vit/output/lsa_convtok/confusion_matrix_slide_lsa_convtok.png
ROC/AUC curve plot saved to: /content/drive/MyDrive/colab/lsa_convtok_vit/output/lsa_convtok/roc_auc_image_lsa_convtok.png
ROC/AUC curve plot saved to: /content/drive/MyDrive/colab/lsa_convtok_vit/output/lsa_convtok/roc_auc_slide_lsa_convtok.png

Results saved: test_results_lsa_convtok.js